#### 현재 위치 확인

In [7]:
%pwd

'c:\\Users\\ADMIN\\Documents\\projects\\team\\Dlthon_02\\main_architecture'

#### 클라이언트 정의

In [8]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()


openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key = openai_api_key)

#### LLM 호출

In [9]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model = "gpt-5.4",
    temperature = 0.6,
    max_tokens = 1000
)

small_llm = ChatOpenAI(
    model = "gpt-5.4-mini",
    temperature = 0.6,
    max_tokens = 1000
)

#### 스테이트 정의

In [10]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph import MessagesState


class AgentState(MessagesState):
    
    # 공용
    response: str
    next: str
    reason: str
    
    # 병인님
    ## 질병 관리
    breed: str
    age: int
    weight: int
    conditions: list[str]
    symptom_raw: str
    hospital: str

    ## 의도파악
    behavior: str
    behavior_cycle: int


    # 아인님 - 서비스 관련한 질문이 들어올 때
    latest_claim: str

#### 슈퍼바이저 노드

In [11]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


class SuperVisor(BaseModel):
    response_reason: str
    next_node: Literal["Agent1", "Agent2", "Agent3", "Agent4", "Agent5"] # = Field(description="다음 실행 노드를 결정")

router_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", """
            당신은 라우터 에이전트입니다.
            대화흐름을 검토하여 다음 사항을 수행하고 그 이유를 간단하게 명시하세요.
            1. 사용자가 반려동물의 상태나 건강에 대해서 물으면 Agent1 노드로 연결하세요.
            2. 사용자가 반려동물의 행동에 대해서 물으면 Agent2 노드로 연결하세요.
            3. 사용자가 서비스에 대해 물으면 Agent3 노드로 연결하세요.
            4. 사용자가 그 외의 것에 대해 질문하면 Agent4 노드로 연결하세요.
            모든 결정에는 간단하고 짧게 이유를 명시하세요."""
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

supervisor_llm  = small_llm.with_structured_output(SuperVisor)
router_chain = router_prompt | supervisor_llm

def supervisor(state: AgentState) -> AgentState:
    response = router_chain.invoke({"messages": state["messages"]})
    
    # 몹시 중요한 부분, 랭그래프의 상태 관리 핵심 (왜 딕셔너리 전체를 리턴하지 않는데 스테이트가 이동하는지?)
    return {
        "next": response.next_node,
        "reason": response.response_reason
    }